### Connecting to Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive/')

Mounted at /content/drive/


### Installing the dependencies

In [ ]:
%%capture
!pip install llama-index
!pip install openai
!pip install pypdf
!pip install llama-index-embeddings-openai
!pip install tiktoken

### Data and persist folder

In [ ]:
import os
import json
import re
import pandas as pd
import openai
import tiktoken

project_folder = "/content/drive/MyDrive/TraCR_RAG_Fresh"
data_folder    = project_folder + "/data"            # the 62 technique files
PERSIST_DIR    = project_folder + "/vector_index"    # the index built in notebook 1
results_folder = project_folder + "/results"

os.makedirs(results_folder, exist_ok=True)

print("Index folder exists:", os.path.exists(PERSIST_DIR))
print("Files in it:", len(os.listdir(PERSIST_DIR)))

Index folder exists: True
Files in it: 5


### Setting up the API

In [ ]:
# The key tells OpenAI who is making the request.
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

print("API key loaded:", os.environ["OPENAI_API_KEY"][:7] + "...")

API key loaded: sk-proj...


### Load the dataset

In [ ]:
dataset_path = project_folder + "/author_repository/data/dataset.jsonl"

with open(dataset_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)

print("Rows:", len(df))
print("Columns:", list(df.columns))

Rows: 433
Columns: ['flow_id', 'str_label', 'TEXT-flow_fn_threat']


### Load the retriever

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Same embedding model that built the index, or the numbers will not match.
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")


def get_retriever(persist_dir, top_k=20):
    """Load the saved index from Drive and return a retriever."""
    print("Loading index from:", persist_dir)
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
    index = load_index_from_storage(storage_context=storage_context)
    print("Index loaded.")

    return VectorIndexRetriever(index=index, similarity_top_k=top_k)


retriever = get_retriever(PERSIST_DIR)

# Quick check that it still works.
nodes = retriever.retrieve("A roadside unit broadcasts messages to vehicles without authentication.")
print("Retrieved", len(nodes), "chunks. Top match:",
      os.path.basename(nodes[0].node.metadata["file_path"]), round(nodes[0].score, 3))

Loading index from: /content/drive/MyDrive/TraCR_RAG_Fresh/vector_index
Index loaded.
Retrieved 20 chunks. Top match: T1557.txt 0.314


### Useful functions

In [ ]:
client = openai.OpenAI()


def count_tokens(text):
    """Count how many tokens a piece of text will cost."""
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")
    return len(encoder.encode(text))


def parse_mitre_techniques(response):
    """Pull every technique ID (like T1234) out of the model's answer."""
    return re.findall(r'T\d{4}', response)


def calculate_metrics(true_labels, predicted_labels):
    """Compare one prediction against the correct answer."""
    true_set = set(true_labels)
    predicted_set = set(predicted_labels)

    tp = len(true_set & predicted_set)     # correct ones it found
    fp = len(predicted_set - true_set)     # ones it invented
    fn = len(true_set - predicted_set)     # ones it missed

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


def generate_prompt(context, question):
    """Their wrapper: retrieved documents on top, the question underneath."""
    return f'''Answer the question using the following description of the information flow from Intelligent Transport System (ITS).
            Choose your answer carefully analyzing the context entirely.
            -----------------------------------------
            {context}
            -----------------------------------------
            Question: {question}
            -----------------------------------------
            Your answer should be a python list:

    '''

### The RAG function

In [ ]:
def get_response(retriever, question, top_k=10, model="gpt-4o-mini-2024-07-18"):
    """Retrieve technique documents, feed them in as context, then ask."""

    nodes = retriever.retrieve(question)

    # Their fixed preamble. Only used here to reserve room in the token budget.
    preamble = '''You are given the description of some information flow from an Intelligent Transport System (ITS). Based on the description and the source and destination of the flow, you have to find out potential MITRE ATT&CK technique that the attacker might use to intervene the flow. Answer in the required format.'''

    contexts = []
    ind = 0
    used = 0

    for node in nodes:
        if ind == len(contexts):
            contexts.append("")

        # One block per retrieved chunk: its text, then which file it came from.
        text  = '----------\n'
        text += 'Description:\n'
        text += node.node.text
        text += 'Source: '
        text += ' (' + os.path.basename(node.node.metadata['file_path'])[:-4] + ')' + "\n\n"
        text += '----------\n'

        # If this block would overflow the model's window, start a new one.
        if count_tokens(contexts[ind] + text + question) + count_tokens(preamble) + 100 + 2000 >= 16385:
            ind += 1
            continue

        contexts[ind] += text

        used += 1
        if used == top_k:
            break

    # Ask once per context block, then join the answers.
    answer = ""
    for context in contexts:
        prompt = generate_prompt(context, question)

        gpt_response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system",
                 "content": '''You are a helpful assisstant. Your name is Transportation Security AI. Your role is to help with transportation security.
                There are some instructions in each prompt. Follow those instructions strictly.'''},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=10000,
            top_p=0,
        )
        answer += str(gpt_response.choices[0].message.content)

    return answer


def build_rag_question(flow_description):
    """Their RAG question wording, kept as they wrote it."""
    return f'''I am trying to do a multilabel classification of information flow description from Intelligent Transportation System (ITS) to MITRE ATT&CK Techniques.
                Here we have information flow name, its source and destination, some functional object description associated with it and the description of the information flow itself. It also has a threat report generated in the STRIDE framework.
                An attacker may attempt to compromise the integrity, confidentiality, or availability of the information flow in many ways.
                Find all the relevant MITRE ATT&CK techniques that the attacker might use to attack the information flow.

                Follow the instructions below carefully.

                1. We have a predefined list of MITRE ATT&CK Techniques that consists of 63 MITRE Techniques. You have to choose only the relevant MITRE ATT&CK Techniques from this list that is relevant to the information flow given.

                2. Understand the entire context, then generate a sublist of MITRE ATT&CK Technique from the given list.

                3. Do not add any other description in your answer.

                4. Only return the Technique IDs in python list format

                Given MITRE Technique List = ['T1495','T1485','T1595','T1134','T1040','T1132','T1098','T1069','T1036','T1562','T1187','T1486','T1119','T1027','T1498','T1654','T1548','T1082','T1552','T1614','T1531','T1204','T1529','T1046','T1489','T1195','T1566','T1659','T1059','T1213','T1133','T1080','T1005','T1078','T1001','T1190','T1203','T1136','T1491','T1033','T1189','T1068','T1652','T1049','T1020','T1041','T1021','T1105','T1518','T1200','T1053','T1557','T1056','T1087','T1565','T1499','T1657','T1559','T1074','T1106','T1560', 'T1556', 'T1589']

                Here is the information flow description:
                {flow_description}

                Which are the relevant MITRE ATT&CK Techniques from the given list that the attacker might use to attack the information flow? Return the Technique IDs in python list format.
                '''

In [ ]:
i = 0
question = build_rag_question(df["TEXT-flow_fn_threat"][i])

print("Question size:", count_tokens(question), "tokens")
print()

answer = get_response(retriever, question)

print("RAW ANSWER:")
print(answer)
print()
print("PARSED:", parse_mitre_techniques(answer))
print("TRUE:  ", df["str_label"][i])

Question size: 1819 tokens

RAW ANSWER:
```python
['T1498', 'T1557', 'T1659', 'T1040', 'T1098', 'T1486']
```

PARSED: ['T1498', 'T1557', 'T1659', 'T1040', 'T1098', 'T1486']
TRUE:   ['T1040', 'T1059', 'T1078', 'T1105', 'T1190', 'T1195', 'T1495', 'T1552', 'T1557', 'T1565']


### Run RAG over the flows

In [ ]:
HOW_MANY = 20

rag_predictions = []

for i in range(HOW_MANY):
    question = build_rag_question(df["TEXT-flow_fn_threat"][i])

    answer = get_response(retriever, question)
    predicted = parse_mitre_techniques(answer)
    rag_predictions.append(predicted)

    print(i, df["flow_id"][i], "->", len(predicted), "techniques")

print()
print("Done.", len(rag_predictions), "flows classified.")

0 2B_TM16-signal_system_configuration -> 6 techniques
1 2A_TM16-traffic_detector_coordination -> 6 techniques
2 2A_TM16-traffic_detector_coordination -> 6 techniques
3 2B_TM16-reversible_lane_control -> 7 techniques
4 2B_TM16-video_surveillance_control -> 6 techniques
5 2B_TM16-signal_control_device_configuration -> 7 techniques
6 _TM16-vehicle_characteristics -> 7 techniques
7 2B_TM16-traffic_detector_control -> 7 techniques
8 2A_TM16-lane_management_coordination -> 7 techniques
9 2A_TM16-lane_management_coordination -> 6 techniques
10 2B_TM16-signal_control_plans -> 6 techniques
11 _TM16-traffic_operator_data -> 4 techniques
12 2A_TM16-signal_control_coordination -> 6 techniques
13 2A_TM16-signal_control_coordination -> 6 techniques
14 2B_TM16-traffic_image_meta_data -> 7 techniques
15 2B_TM16-reversible_lane_status -> 10 techniques
16 _PT04-traveler_interface_updates -> 3 techniques
17 2C_PT09-traffic_control_priority_status -> 8 techniques
18 2C_PT09-traffic_control_priority_reques

### Score the RAG run

In [ ]:
rag_results = []

for i in range(HOW_MANY):
    metrics = calculate_metrics(df["str_label"][i], rag_predictions[i])
    rag_results.append(metrics)

avg_predicted = sum(len(p) for p in rag_predictions) / HOW_MANY
avg_true = sum(len(df["str_label"][i]) for i in range(HOW_MANY)) / HOW_MANY
print("Average predicted per flow: %.1f" % avg_predicted)
print("Average true per flow:      %.1f" % avg_true)
print()

total_tp = sum(r["tp"] for r in rag_results)
total_fp = sum(r["fp"] for r in rag_results)
total_fn = sum(r["fn"] for r in rag_results)

rag_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
rag_recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
rag_f1 = 2 * rag_precision * rag_recall / (rag_precision + rag_recall) if (rag_precision + rag_recall) > 0 else 0

print("Correct found (TP):", total_tp)
print("Invented (FP):    ", total_fp)
print("Missed (FN):      ", total_fn)
print()
print("Precision: %.3f" % rag_precision)
print("Recall:    %.3f" % rag_recall)
print("F1:        %.3f" % rag_f1)

Average predicted per flow: 6.4
Average true per flow:      7.8

Correct found (TP): 32
Invented (FP):     96
Missed (FN):       124

Precision: 0.250
Recall:    0.205
F1:        0.225


### The expert review set

In [ ]:
expert_path = project_folder + "/author_repository/data/50_random_flows.json"

with open(expert_path, "r", encoding="utf-8") as f:
    expert_data = json.load(f)

print("Flows reviewed:", len(expert_data))
print()

added_total = 0
removed_total = 0
unchanged = 0

for row in expert_data:
    original = set(row["y_true"])
    reviewed = set(row["y_true_expert"])

    added = reviewed - original      # techniques the experts put in
    removed = original - reviewed    # techniques the experts took out

    added_total += len(added)
    removed_total += len(removed)
    if not added and not removed:
        unchanged += 1

print("Labels experts ADDED:  ", added_total)
print("Labels experts REMOVED:", removed_total)
print("Flows left unchanged:  ", unchanged, "of", len(expert_data))
print()
print("Average labels before:", round(sum(len(r["y_true"]) for r in expert_data) / len(expert_data), 1))
print("Average labels after: ", round(sum(len(r["y_true_expert"]) for r in expert_data) / len(expert_data), 1))

Flows reviewed: 50

Labels experts ADDED:   109
Labels experts REMOVED: 0
Flows left unchanged:   4 of 50

Average labels before: 12.7
Average labels after:  14.9


### Line up the reviewed flows with the dataset

In [ ]:
# Match each reviewed flow to its row number.
row_of = {}
for i in range(len(df)):
    flow_id = df["flow_id"][i]
    if flow_id not in row_of:        # one flow_id appears twice, keep the first
        row_of[flow_id] = i

reviewed = []
for entry in expert_data:
    i = row_of.get(entry["flow_id"])
    if i is not None:
        reviewed.append({
            "row": i,
            "flow_id": entry["flow_id"],
            "y_true": entry["y_true"],
            "y_true_expert": entry["y_true_expert"],
        })

print("Reviewed flows matched to rows:", len(reviewed))

# Confirm the dataset carries the ORIGINAL labels, not the reviewed ones.
disagree = sum(1 for r in reviewed if set(df["str_label"][r["row"]]) != set(r["y_true"]))
print("Rows where dataset labels disagree with y_true:", disagree, "(should be 0)")

Reviewed flows matched to rows: 50
Rows where dataset labels disagree with y_true: 0 (should be 0)


### Run RAG on the 50 reviewed flows

In [ ]:
for r in reviewed:
    question = build_rag_question(df["TEXT-flow_fn_threat"][r["row"]])

    answer = get_response(retriever, question)
    r["predicted"] = parse_mitre_techniques(answer)

    print(r["row"], r["flow_id"], "->", len(r["predicted"]), "techniques")

print()
print("Done.", len(reviewed), "flows classified.")

324 2B_PT09-right_of_way_request_notification -> 6 techniques
153 2B_PT10-transit_schedule_information -> 7 techniques
181 2C_CVO05-fleet_to_driver_update -> 5 techniques
331 2C_PS11-transit_emergency_data -> 6 techniques
209 2C_PT04-payment_request -> 5 techniques
211 2C_PT04-transit_fare_request -> 8 techniques
160 2C_PT14-event_transit_service_plans -> 4 techniques
296 2C_TM15-rail_crossing_advisories -> 5 techniques
346 2Ca_PS11-incident_information -> 5 techniques
137 _CVO15-alert_response -> 7 techniques
92 _PS11-transit_operations_personnel_input -> 7 techniques
39 _SU09-credentials_management_operator_presentation -> 7 techniques
242 1A_CVO15-cv_driver_credential -> 8 techniques
74 2A_PT03-transit_vehicle_operator_information -> 6 techniques
239 2B_CVO15-route_deviation_alert -> 7 techniques
151 2B_PS11-secure_area_surveillance_control -> 7 techniques
282 2B_PS14-evacuee_match_information -> 4 techniques
77 2B_TM24-traffic_metering_control -> 6 techniques
238 2C_CVO15-commercia

### Score the same predictions against both answer keys

In [ ]:
def score_against(records, key):
    """Score every prediction against one version of the answer key."""
    tp = fp = fn = 0
    for r in records:
        m = calculate_metrics(r[key], r["predicted"])
        tp += m["tp"]
        fp += m["fp"]
        fn += m["fn"]

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    rc = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * rc / (p + rc) if (p + rc) > 0 else 0
    return tp, fp, fn, p, rc, f


# The predictions never change. Only the answer key does.
before = score_against(reviewed, "y_true")
after  = score_against(reviewed, "y_true_expert")

print("Predictions are identical in both rows below.")
print("The only thing that changes is which answer key they are scored against.")
print()
print("%-22s %5s %5s %5s %10s %8s %8s" % ("Answer key", "TP", "FP", "FN", "Precision", "Recall", "F1"))
print("%-22s %5d %5d %5d %10.3f %8.3f %8.3f" % ("Original (y_true)", *before))
print("%-22s %5d %5d %5d %10.3f %8.3f %8.3f" % ("Reviewed (expert)", *after))
print()
print("Precision moved: %+.3f" % (after[3] - before[3]))
print("False positives that became true positives:", before[1] - after[1])

Predictions are identical in both rows below.
The only thing that changes is which answer key they are scored against.

Answer key                TP    FP    FN  Precision   Recall       F1
Original (y_true)        115   187   520      0.381    0.181    0.245
Reviewed (expert)        141   161   603      0.467    0.190    0.270

Precision moved: +0.086
False positives that became true positives: 26
